# Colab 02 - Extract latents

This notebook extracts MMAudio training latents for each split and stores the TensorDict memmaps in Google Drive.

It expects videos in:

```text
/content/drive/MyDrive/MMAudio_Yuqi/data/mmaudio_dataset_stride8/train
/content/drive/MyDrive/MMAudio_Yuqi/data/mmaudio_dataset_stride8/val
/content/drive/MyDrive/MMAudio_Yuqi/data/mmaudio_dataset_stride8/test
```

Each split gets its own memmap directory. This matters because the current training dataset indexes memmaps by row position.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, '/content/MMAudio')

from fine_tune.urban_paths import (
    AUDIO_SR,
    CLIP_SEQ_LEN,
    DEFAULT_STRIDE,
    DRIVE_ROOT,
    EVAL_DIR,
    EXT_WEIGHTS_DIR,
    LATENT_SEQ_LEN,
    LOG_DIR,
    MODE,
    MODEL,
    OUTPUT_DIR,
    REPO_ROOT,
    SYNC_SEQ_LEN,
    TEXT_LABEL,
    TEXT_SEQ_LEN,
    TSV_DIR,
    VIDEO_EXTENSIONS,
    WEIGHTS_DIR,
    ensure_drive_dirs,
    latent_dir_for_stride,
    split_video_dirs,
)

STRIDE = DEFAULT_STRIDE
SPLIT_VIDEO_DIRS = split_video_dirs(STRIDE)
LATENTS_ROOT = latent_dir_for_stride(STRIDE)
RUNS_ROOT = OUTPUT_DIR
EVAL_ROOT = EVAL_DIR

ensure_drive_dirs(STRIDE)
for split_dir in SPLIT_VIDEO_DIRS.values():
    split_dir.mkdir(parents=True, exist_ok=True)

print('Repo:', REPO_ROOT)
print('Drive root:', DRIVE_ROOT)
print('Stride:', STRIDE)
print('Mode/model:', MODE, MODEL)
print('Latents:', LATENTS_ROOT)
print('Outputs:', RUNS_ROOT)
print('Eval:', EVAL_ROOT)
print('Logs:', LOG_DIR)

In [ ]:
%cd /content/MMAudio

import pandas as pd


def list_videos(root: Path):
    if not root.exists():
        raise FileNotFoundError(f'Missing video directory: {root}')
    videos = [p for p in root.iterdir() if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS]
    videos = sorted(set(videos), key=lambda p: str(p))
    if not videos:
        raise FileNotFoundError(f'No videos found in {root}')
    return videos


def write_split_tsv(split: str, video_root: Path, caption: str = TEXT_LABEL):
    videos = list_videos(video_root)
    rows = [{'id': p.stem, 'caption': caption} for p in videos]
    out_path = TSV_DIR / f'video_{split}_ft.tsv'
    pd.DataFrame(rows).to_csv(out_path, sep='\t', index=False)
    print(f'{split}: wrote {len(rows)} rows to {out_path}')
    return out_path

split_tsvs = {
    split: write_split_tsv(split, root)
    for split, root in SPLIT_VIDEO_DIRS.items()
}

In [ ]:
%cd /content/MMAudio

import os
import subprocess

RUN_SPLITS = ['train', 'val', 'test']
BATCH_SIZE = 1
NUM_WORKERS = 0
DEBUG_LIMIT = None       # Set to a small integer for a smoke test, then back to None.
OVERWRITE = False        # Set True only when you intentionally want to re-extract a split.

for split in RUN_SPLITS:
    video_dir = SPLIT_VIDEO_DIRS[split]
    latent_parent = LATENTS_ROOT / split
    memmap_dir = latent_parent / 'training_latents'

    if memmap_dir.exists() and not OVERWRITE:
        print(f'Skipping {split}; found existing memmap at {memmap_dir}')
        continue

    latent_parent.mkdir(parents=True, exist_ok=True)

    cmd = [
        'python', 'fine_tune/extract_video_training_latents_colab.py',
        '--video_dir', str(video_dir),
        '--latent_dir', str(latent_parent),
        '--save_name', 'training_latents',
        '--mode', MODE,
        '--audio_sr', str(AUDIO_SR),
        '--vae_path', str(EXT_WEIGHTS_DIR / 'v1-16.pth'),
        '--bigvgan_path', str(EXT_WEIGHTS_DIR / 'best_netG.pt'),
        '--synchformer_ckpt', str(EXT_WEIGHTS_DIR / 'synchformer_state_dict.pth'),
        '--batch_size', str(BATCH_SIZE),
        '--num_workers', str(NUM_WORKERS),
    ]
    if DEBUG_LIMIT is not None:
        cmd.extend(['--debug_limit', str(DEBUG_LIMIT)])

    print('\nRunning:', ' '.join(cmd))
    env = os.environ.copy()
    env['OMP_NUM_THREADS'] = '2'
    subprocess.run(cmd, check=True, env=env)

In [ ]:
%cd /content/MMAudio

import json
import pandas as pd
from tensordict import TensorDict

for split in ['train', 'val', 'test']:
    tsv_path = TSV_DIR / f'video_{split}_ft.tsv'
    memmap_dir = LATENTS_ROOT / split / 'training_latents'
    meta_path = memmap_dir / 'metadata.json'

    if not memmap_dir.exists():
        raise FileNotFoundError(f'Missing memmap for {split}: {memmap_dir}')

    td = TensorDict.load_memmap(memmap_dir)
    n_rows = len(pd.read_csv(tsv_path, sep='\t'))

    shapes = {key: tuple(td[key].shape) for key in td.keys()}
    print(f'\n{split}')
    print('TSV rows:', n_rows)
    print('Shapes:', shapes)

    assert shapes['mean'][0] == n_rows, (split, shapes['mean'][0], n_rows)
    assert shapes['std'][0] == n_rows, (split, shapes['std'][0], n_rows)
    assert shapes['mean'][1] == LATENT_SEQ_LEN, shapes['mean']
    assert shapes['std'][1] == LATENT_SEQ_LEN, shapes['std']
    assert shapes['clip_features'][1] == CLIP_SEQ_LEN, shapes['clip_features']
    assert shapes['sync_features'][1] == SYNC_SEQ_LEN, shapes['sync_features']
    assert shapes['text_features'][1] == TEXT_SEQ_LEN, shapes['text_features']

    if meta_path.exists():
        metadata = json.loads(meta_path.read_text())
        print('Written samples:', metadata.get('written_samples'))
        print('Failed batches:', metadata.get('num_failed_batches'))